# Your First RNA-seq Analysis — in R with DESeq2 ★

_BioNexus Hub — bionexushub.com · Runnable companion to the flagship lesson._

**Set the runtime to R first:** Runtime → Change runtime type → R. Then run each cell with Shift+Enter.


## 1. Set up
Installing Bioconductor packages takes a few minutes the first time — grab a coffee.


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("DESeq2", update = FALSE, ask = FALSE)


In [ ]:
library(DESeq2)


## 2. Load the data
A small example count matrix (genes × samples) and a metadata table with a `condition` column.


In [ ]:
base <- "https://raw.githubusercontent.com/owkin/PyDESeq2/main/datasets/synthetic/"

# Count matrix: rows = genes, columns = samples — exactly what DESeq2 wants
counts <- as.matrix(read.csv(paste0(base, "test_counts.csv"), row.names = 1))

# Metadata: one row per sample, with a "condition" column (A / B)
metadata <- read.csv(paste0(base, "test_metadata.csv"), row.names = 1)
metadata$condition <- factor(metadata$condition)

# Keep metadata rows in the same order as the count columns
metadata <- metadata[colnames(counts), , drop = FALSE]

dim(counts)        # genes x samples
head(counts)


## 3. Run the analysis
DESeq2 normalizes the data and fits the statistical model. `design = ~condition` is the question we're asking.


In [ ]:
dds <- DESeqDataSetFromMatrix(countData = counts,
                              colData   = metadata,
                              design    = ~ condition)

dds <- DESeq(dds)   # fit the model


## 4. Read the results
For every gene: is the difference between B and A big and consistent enough to be real? Trust **padj** (adjusted p-value), not the raw pvalue.


In [ ]:
res <- results(dds, contrast = c("condition", "B", "A"))
res <- res[order(res$padj), ]
head(res, 10)


## 5. Visualize: the volcano plot
Each dot is a gene. Big change *and* high confidence fly into the top corners.


In [ ]:
df <- as.data.frame(res)
df <- df[!is.na(df$padj), ]
df$sig <- df$padj < 0.05

plot(df$log2FoldChange, -log10(df$padj),
     pch = 20,
     col = ifelse(df$sig, "#a855f7", "#cbd5e1"),
     xlab = "log2 fold change (B vs A)",
     ylab = "-log10 adjusted p-value",
     main = "Volcano plot — your first one!")
abline(h = -log10(0.05), lty = 2, col = "#64748b")


### Next steps
Swap in a real dataset (e.g. the **airway** experiment, GEO **GSE52778**) — only the data-loading step changes; everything else is identical. In R you can even `BiocManager::install("airway")` to get it ready-made.

**Prefer Python?** The same analysis runs with **PyDESeq2** (`pip install pydeseq2`) — a faithful Python re-implementation. See the lesson page for the Python code.
